# Character-level LM on WikiText-2 (fixed)
Embedding + stacked BiLSTM predicting the next character from the previous 100.

In [1]:
!pip install -q tensorflow datasets

In [2]:
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [3]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [4]:
# ---- Config: raise these once everything works and you have a GPU ----
MAX_CHARS   = 500_000   # how much of WikiText to use (full train split is ~10.9M chars)
SEQ_LENGTH  = 100       # context window (characters)
N_SAMPLES   = 100_000   # number of random training windows
EPOCHS      = 15
BATCH_SIZE  = 128

# Drop empty lines, join, and collapse repeated whitespace
lines = [t.strip() for t in ds["train"]["text"] if t.strip()]
text = " ".join(lines)
text = " ".join(text.split())
text = text[:MAX_CHARS]
print(f"Text length: {len(text):,} characters")
print(text[:300])

Text length: 500,000 characters
= Valkyria Chronicles III = Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation 


In [5]:
chars = sorted(set(text))
mapping = {c: i for i, c in enumerate(chars)}
inverse_mapping = {i: c for c, i in mapping.items()}
vocab_size = len(mapping)
print(f"Vocabulary size (unique characters): {vocab_size}")

Vocabulary size (unique characters): 169


In [6]:
# Encode the whole text once as an integer array
data = np.array([mapping[c] for c in text], dtype=np.int32)

# Sliding windows as a *view* (no memory blow-up). Each row = 100 inputs + 1 target.
windows = np.lib.stride_tricks.sliding_window_view(data, SEQ_LENGTH + 1)
print("Possible windows:", windows.shape[0])

# Randomly pick N_SAMPLES windows (copying only those)
n = min(N_SAMPLES, windows.shape[0])
idx = np.random.choice(windows.shape[0], size=n, replace=False)
sequences = windows[idx]                      # (n, 101)

X, y = sequences[:, :-1], sequences[:, -1]    # X: (n, 100), y: (n,)
print("X:", X.shape, "y:", y.shape)

Possible windows: 499900
X: (100000, 100) y: (100000,)


In [7]:
def create_model(vocab_size, seq_length):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=64),   # no input_length (removed in Keras 3)
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(128)),
        Dropout(0.2),
        Dense(vocab_size, activation="softmax"),
    ])
    model.build(input_shape=(None, seq_length))
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer="adam", metrics=["accuracy"])
    model.summary()
    return model

model = create_model(vocab_size, SEQ_LENGTH)

history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │        10,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 100, 256)       │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 169)            │        43,433 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 646,121 (2.46 MB)

 Trainable params: 646,121 (2.46 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1197s 2s/step - accuracy: 0.2532 - loss: 2.7957 - val_accuracy: 0.3076 - val_loss: 2.4862
Epoch 2/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1190s 2s/step - accuracy: 0.3136 - loss: 2.4326 - val_accuracy: 0.3447 - val_loss: 2.3128
Epoch 3/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1189s 2s/step - accuracy: 0.3458 - loss: 2.3000 - val_accuracy: 0.3614 - val_loss: 2.2160
Epoch 4/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1225s 2s/step - accuracy: 0.3635 - loss: 2.2208 - val_accuracy: 0.3800 - val_loss: 2.1525
Epoch 5/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1190s 2s/step - accuracy: 0.3774 - loss: 2.1607 - val_accuracy: 0.3938 - val_loss: 2.0964
Epoch 6/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1208s 2s/step - accuracy: 0.3930 - loss: 2.1082 - val_accuracy: 0.4047 - val_loss: 2.0467
Epoch 7/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1224s 2s/step - accuracy: 0.4049 - loss: 2.0638 - val_accuracy: 0.4193 - val_loss: 2.0093
Epoch 8/15
704/704 ━━━━━━━━━━━━━━━━━━━━ 1214s 2s/step - accuracy: 0.4148 - loss: 2.0242 - 

In [8]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    probs = np.exp(preds - preds.max())
    probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)


def generate_sequence(model, mapping, max_length, seed_text, n_chars, temperature=0.7):
    inverse_mapping = {i: c for c, i in mapping.items()}
    pad_idx = mapping.get(" ", 0)                 # pad short seeds on the left with a space
    out = seed_text
    for _ in range(n_chars):
        encoded = [mapping.get(c, pad_idx) for c in out[-max_length:]]   # unknown chars -> space
        encoded = [pad_idx] * (max_length - len(encoded)) + encoded
        probs = model.predict(np.array([encoded]), verbose=0)[0]
        out += inverse_mapping[sample(probs, temperature)]
    return out


for t in (0.5, 0.8, 1.0):
    print(f"--- temperature {t} ---")
    print(generate_sequence(model, mapping, SEQ_LENGTH, "the scientific method involves", 300, t))
    print()

--- temperature 0.5 ---
the scientific method involves as the more flow @-@ hourded as the recined for the secile as a samed , designed destented the sast of the sont . In the world of the precation and the recold that the gods of the for of the Ealk and has lill regaration . He as in the asson to pesson . The sited and the sited . The south was was de

--- temperature 0.8 ---
the scientific method involves Uadelowing surmadions of coothes by the shian , Near 1997 . = = = Thesual to 1868 . The flackes is a ghele alishiagland , the storsher conning to Belitoly . " The standed meve coverated than the harkainalul a coup considengs deepigions mive domition by she patters . The gorth the sisters . The gold

--- temperature 1.0 ---
the scientific method involves of Morin to the fountravaration , contrues , were stendeling enegroched his mange had as ovondelf deave and nome of the LlocA Flided Ileotion the Unuctor = = = = = Cheen @-@ which itply 20 @-@ tersiton Psuption issuvall froks horimes

In [9]:
for t in (0.5, 0.8, 1.0):
    print(f"--- temperature {t} ---")
    print(generate_sequence(model, mapping, SEQ_LENGTH, "the neuroscience was always associated with", 300, t))
    print()

--- temperature 0.5 ---
the neuroscience was always associated with the fork of the down . The his become . = = = Theses as the seven as a sound bacted the becord to commention sited as batt the fill designed are becaral repurted about on the song of the startions . The procouth and religion to the the recond of the game is has are the partions . The senge to the c

--- temperature 0.8 ---
the neuroscience was always associated with presend an lock coltainaly mit and @-@ the Eation Materral Lamereak , where had on the stracle by socond projector be Mew undivered album album . These with the scorded along tiffer to the some enferary the Pought as the has oflan that the eplosed in they of the sent coner , and oN and deconding to

--- temperature 1.0 ---
the neuroscience was always associated with the sipt of the bit bedore moses interved . Fend or Nash in scods ofoles that asrussion to a some diseral seatuas fucys that to preciefs by houss and gom of the Trysist life iits . It have locem